# ACTIVE-TB-X v7 - Exact LLM Contribution Ablation

This focused follow-up compares two otherwise identical methods on the same 24 VerilogEval tasks and the same 32-cycle budget:

1. `active_portfolio_no_llm`: temporal objectives + portfolio scheduler + BMC + heuristics + real coverage,
2. `active_portfolio`: the identical pipeline with evidence-gated Qwen proposals enabled.

The paired design isolates the LLM contribution. Mutation banks, visible/hidden splits, seeds, budgets, task order, formal triage, and scoring remain unchanged. Every `(task, method)` result is checkpointed independently.


In [ ]:
# 1) GPU + EDA tools. Do not upgrade Kaggle's base Python stack.
import os, shutil, subprocess
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

subprocess.run(["nvidia-smi", "-L"], check=True)
missing = [x for x in ["iverilog", "verilator", "verilator_coverage", "yosys", "yosys-smtbmc", "z3"] if shutil.which(x) is None]
if missing:
    print("Installing EDA tools because these are missing:", missing)
    subprocess.run("apt-get update -qq >/tmp/active_tb_apt_update.log 2>&1", shell=True, check=True)
    subprocess.run("DEBIAN_FRONTEND=noninteractive apt-get install -y -qq iverilog verilator yosys z3 >/tmp/active_tb_apt_install.log 2>&1", shell=True, check=True)
for tool in ["iverilog", "vvp", "verilator", "verilator_coverage", "yosys", "yosys-smtbmc", "z3"]:
    path = shutil.which(tool)
    print(f"{tool}: {path}")
    if path is None:
        raise RuntimeError(f"Required EDA tool missing: {tool}")
print("EDA_PREFLIGHT_PASS")


In [ ]:
# 2) Locate/copy this project to writable /kaggle/working.
from pathlib import Path
import shutil, os, zipfile

WORK_PROJECT = Path("/kaggle/working/active_tb_plus_project")
candidates = []
for base in [Path.cwd(), Path("/kaggle/input")]:
    if not base.exists():
        continue
    for p in base.rglob("run_budget_sweep.py"):
        if (p.parent / "active_tb" / "engine.py").exists():
            candidates.append(p.parent)
if not candidates:
    # Kaggle datasets often carry the Windows-built project as one portable
    # archive. Extract only after checking every member stays below the target.
    extract_root = Path("/kaggle/working/active_tb_plus_attached_source")
    archives = sorted(
        p for p in Path("/kaggle/input").rglob("*.zip")
        if "active_tb" in p.name.lower() or "active-tb" in p.name.lower()
    )
    for archive in archives:
        if extract_root.exists():
            shutil.rmtree(extract_root)
        extract_root.mkdir(parents=True)
        try:
            with zipfile.ZipFile(archive) as zf:
                for member in zf.infolist():
                    target = (extract_root / member.filename).resolve()
                    if target != extract_root.resolve() and extract_root.resolve() not in target.parents:
                        raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
                zf.extractall(extract_root)
            for p in extract_root.rglob("run_budget_sweep.py"):
                if (p.parent / "active_tb" / "engine.py").exists():
                    candidates.append(p.parent)
        except zipfile.BadZipFile:
            continue
        if candidates:
            print("Extracted project archive:", archive)
            break
if not candidates:
    raise RuntimeError("Could not find ACTIVE-TB+ project directory or portable ZIP in Kaggle inputs.")
source = sorted(candidates, key=lambda p: (0 if str(p).startswith('/kaggle/input') else 1, len(str(p))))[0]
print("Project source:", source)
if source.resolve() != WORK_PROJECT.resolve():
    if WORK_PROJECT.exists():
        shutil.rmtree(WORK_PROJECT)
    shutil.copytree(source, WORK_PROJECT, ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "outputs"))
os.chdir(WORK_PROJECT)
print("Working project:", Path.cwd())
assert (WORK_PROJECT / "run_budget_sweep.py").exists()


In [ ]:
# 3) Build the isolated ACTIVE-TB environment without ensurepip.
import subprocess, sys
from pathlib import Path
VENV = Path("/kaggle/working/active_tb_venv")
VENV_PY = VENV / "bin" / "python"
cmd = [
    sys.executable, str(WORK_PROJECT / "bootstrap_kaggle.py"),
    "--venv", str(VENV),
    "--requirements", str(WORK_PROJECT / "requirements.txt"),
]
print("Bootstrap interpreter:", sys.executable)
subprocess.run(cmd, check=True, cwd=WORK_PROJECT)
assert VENV_PY.exists()
print("VENV_PY =", VENV_PY)


In [ ]:
# 4) Dependency/Qwen import + T4 x2 preflight.
import subprocess
subprocess.run([str(VENV_PY), str(WORK_PROJECT / "environment_smoke_test.py")], check=True, cwd=WORK_PROJECT)
subprocess.run([str(VENV_PY), str(WORK_PROJECT / "check_kaggle_runtime.py")], check=True, cwd=WORK_PROJECT)


In [ ]:
# 5) Discover Kaggle-mounted Qwen3.5-27B.
from pathlib import Path
import json
MANUAL_QWEN_MODEL_PATH = None

def qwen_candidates(root="/kaggle/input"):
    found=[]
    root=Path(root)
    if not root.exists(): return found
    for cfg in root.rglob("config.json"):
        d=cfg.parent
        try: meta=json.loads(cfg.read_text())
        except Exception: meta={}
        text=(str(d)+" "+str(meta.get("model_type", ""))+" "+" ".join(meta.get("architectures", []) or [])).lower()
        has_weights = any(d.glob("*.safetensors")) or any(d.glob("pytorch_model*.bin"))
        if has_weights and ("qwen3_5" in text or "qwen3.5" in text or "qwen3-5" in text):
            found.append(d)
    return sorted(set(found), key=lambda p: (0 if "27b" in str(p).lower() else 1, len(str(p))))

cands=qwen_candidates()
print("Qwen3.5 model candidates:")
for i,p in enumerate(cands): print(f"[{i}] {p}")
if MANUAL_QWEN_MODEL_PATH:
    QWEN_MODEL_PATH=Path(MANUAL_QWEN_MODEL_PATH)
elif len(cands)==1:
    QWEN_MODEL_PATH=cands[0]
elif cands and "27b" in str(cands[0]).lower():
    QWEN_MODEL_PATH=cands[0]
else:
    raise RuntimeError("Could not choose Qwen3.5-27B automatically; set MANUAL_QWEN_MODEL_PATH.")
assert (QWEN_MODEL_PATH / "config.json").exists()
print("QWEN_MODEL_PATH =", QWEN_MODEL_PATH)


In [ ]:
# 6) Optional full model-placement preflight. This loads then releases Qwen once.
# It costs a few minutes but catches 2xT4 placement failures before a long run.
RUN_FULL_MODEL_LOAD_PREFLIGHT = True
if RUN_FULL_MODEL_LOAD_PREFLIGHT:
    import subprocess
    subprocess.run([
        str(VENV_PY), str(WORK_PROJECT / "check_kaggle_runtime.py"),
        "--load-model", "--model-path", str(QWEN_MODEL_PATH),
        "--per-gpu-memory-gb", "13.25",
    ], check=True, cwd=WORK_PROJECT)


In [ ]:
# 7) Static + checkpoint + adaptive + budget-report smoke tests.
import subprocess
for script in [
    "smoke_test.py",
    "checkpoint_smoke_test.py",
    "adaptive_smoke_test.py",
    "portfolio_smoke_test.py",
    "real_coverage_smoke_test.py",
    "mutation_balance_smoke_test.py",
    "temporal_coverage_smoke_test.py",
    "formal_triage_smoke_test.py",
    "formal_integration_smoke_test.py",
    "budget_report_smoke_test.py",
    "budget_method_smoke_test.py",
]:
    subprocess.run([str(VENV_PY), str(WORK_PROJECT / script)], check=True, cwd=WORK_PROJECT)
print("ACTIVE_TB_PLUS_SMOKE_PASS")


In [ ]:
# 8) Optional local VerilogEval dataset discovery.
from pathlib import Path
MANUAL_LOCAL_DATASET = None
if MANUAL_LOCAL_DATASET:
    LOCAL_DATASET = MANUAL_LOCAL_DATASET
else:
    likely=[]
    for ext in ("*.parquet", "*.jsonl", "*.json"):
        for p in Path("/kaggle/input").rglob(ext):
            if "verilog" in str(p).lower() and "eval" in str(p).lower():
                likely.append(p)
    LOCAL_DATASET = str(sorted(likely, key=lambda p: len(str(p)))[0]) if likely else None
print("LOCAL_DATASET =", LOCAL_DATASET)


In [ ]:
# 9) Checkpoints: keep these two roles separate.
from pathlib import Path

# SAME new budget-sweep experiment only. Use this to resume after a Kaggle session ends.
BUDGET_SWEEP_CHECKPOINT = None

# OPTIONAL old final 64-cycle run. Only contract.json + golden.sv are reused.
# Never use the old checkpoint as BUDGET_SWEEP_CHECKPOINT.
OLD_FULL_RUN_CHECKPOINT = None

def checkpoint_candidates(root="/kaggle/input"):
    root=Path(root); out=[]
    if not root.exists(): return out
    out += [p for p in root.rglob("*checkpoint*.zip") if "active_tb" in str(p).lower()]
    out += [p.parent for p in root.rglob("per_task_results.csv") if "active_tb" in str(p).lower()]
    return sorted(set(map(str,out)))

print("Checkpoint candidates:")
for i,p in enumerate(checkpoint_candidates()): print(f"[{i}] {p}")
print("BUDGET_SWEEP_CHECKPOINT =", BUDGET_SWEEP_CHECKPOINT)
print("OLD_FULL_RUN_CHECKPOINT =", OLD_FULL_RUN_CHECKPOINT)


In [ ]:
# 10) Fast exact-budget NO-LLM integration smoke.
# Confirms the new method, strict budgets, repeated Random seeds, BMC, and reports.
import subprocess
SMOKE_OUT = "/kaggle/working/active_tb_plus_budget_smoke"
cmd = [
    str(VENV_PY), str(WORK_PROJECT / "run_budget_sweep.py"),
    "--output", SMOKE_OUT,
    "--limit", "3",
    "--budgets", "8", "16",
    "--methods", "seed_only", "random", "greedy_mutation", "bmc_only", "active_no_llm", "active_adaptive", "active_portfolio_no_llm", "active_portfolio",
    "--random-seeds", "0", "1",
    "--bmc-timeout", "10",
    "--session-hours", "0",
    "--snapshot-every-jobs", "10",
    "--bootstrap-resamples", "200",
]
if LOCAL_DATASET:
    cmd += ["--local-dataset", LOCAL_DATASET]
print(" ".join(map(str,cmd)))
subprocess.run(cmd, check=True, cwd=WORK_PROJECT)


In [ ]:
# 11) Small LLM + real-coverage smoke for ACTIVE-TB-X.
RUN_LLM_SMOKE = True
if RUN_LLM_SMOKE:
    import subprocess
    LLM_SMOKE_OUT = "/kaggle/working/active_tb_plus_llm_smoke"
    cmd = [
        str(VENV_PY), str(WORK_PROJECT / "run_budget_sweep.py"),
        "--output", LLM_SMOKE_OUT,
        "--limit", "2",
        "--budgets", "16",
        "--methods", "active_portfolio",
        "--random-seeds", "0",
        "--enable-llm", "--model-path", str(QWEN_MODEL_PATH),
        "--per-gpu-memory-gb", "13.25",
        "--max-input-tokens", "2560",
        "--llm-batch-candidates", "4",
        "--enable-real-coverage", "--real-coverage-interval", "1",
        "--session-hours", "0",
        "--snapshot-every-jobs", "5",
        "--bootstrap-resamples", "100",
    ]
    if LOCAL_DATASET:
        cmd += ["--local-dataset", LOCAL_DATASET]
    if OLD_FULL_RUN_CHECKPOINT:
        cmd += ["--reuse-contracts-from", OLD_FULL_RUN_CHECKPOINT]
    print(" ".join(map(str,cmd)))
    subprocess.run(cmd, check=True, cwd=WORK_PROJECT)


In [ ]:
# 12) ACTIVE-TB-X exact LLM attribution run.
RUN_LLM_ABLATION = True
if RUN_LLM_ABLATION:
    import subprocess
    OUT = "/kaggle/working/active_tb_x_llm_ablation_v7"
    cmd = [
        str(VENV_PY), str(WORK_PROJECT / "run_budget_sweep.py"),
        "--output", OUT,
        "--limit", "24",
        "--task-start", "0", "--task-end", "156",
        "--budgets", "32",
        "--methods", "active_portfolio_no_llm", "active_portfolio",
        "--random-seeds", "0",
        "--enable-llm", "--model-path", str(QWEN_MODEL_PATH),
        "--per-gpu-memory-gb", "13.25",
        "--max-input-tokens", "2560",
        "--bmc-timeout", "15",
        "--max-bmc-calls", "8",
        "--max-llm-calls", "4",
        "--adaptive-max-rounds", "20",
        "--exploration-pool", "20",
        "--max-exploration-additions", "6",
        "--llm-stagnation-rounds", "2",
        "--coverage-weight", "0.75",
        "--functional-coverage-weight", "1.0",
        "--novelty-weight", "0.35",
        "--family-weight", "0.50",
        "--llm-batch-candidates", "4",
        "--scheduler-exploration-strength", "0.65",
        "--enable-real-coverage",
        "--real-coverage-interval", "2",
        "--real-coverage-timeout", "90",
        "--mutant-triage-timeout", "4",
        "--max-mutants-per-family", "4",
        "--max-mutants-total", "48",
        "--session-hours", "10.25",
        "--stop-before-minutes", "45",
        "--snapshot-every-jobs", "4",
        "--bootstrap-resamples", "2000",
        "--snapshot-path", "/kaggle/working/active_tb_x_llm_ablation_v7_checkpoint_latest.zip",
    ]
    if LOCAL_DATASET:
        cmd += ["--local-dataset", LOCAL_DATASET]
    if BUDGET_SWEEP_CHECKPOINT:
        cmd += ["--resume-from", BUDGET_SWEEP_CHECKPOINT]
    if OLD_FULL_RUN_CHECKPOINT:
        cmd += ["--reuse-contracts-from", OLD_FULL_RUN_CHECKPOINT]
    print(" ".join(map(str, cmd)))
    subprocess.run(cmd, check=True, cwd=WORK_PROJECT)


In [ ]:
# 13) Enforce invariants and report exact LLM attribution.
from collections import Counter
from pathlib import Path
import json
import pandas as pd

def validate_sanity_run(root, expected_rows, allowed_budgets, require_llm=False):
    root = Path(root)
    results_path = root / "per_task_results.csv"
    assert results_path.exists(), f"Missing results: {results_path}"
    df = pd.read_csv(results_path)
    assert len(df) == expected_rows, f"Expected {expected_rows} rows in {root}, found {len(df)}"
    assert not df.duplicated(["task_id", "method"]).any(), f"Duplicate jobs in {root}"
    parsed_budget = pd.to_numeric(df["method"].astype(str).str.extract(r"^b(\d+)__", expand=False), errors="coerce")
    assert parsed_budget.notna().all(), f"Unparseable exact-budget method labels in {root}"
    assert set(parsed_budget.astype(int)).issubset(set(allowed_budgets))
    supported = df["supported"].astype(str).str.lower().eq("true")
    assert supported.all(), f"Unsupported jobs in {root}:\n{df.loc[~supported, ['task_id', 'method', 'reason']]}"
    over = df.loc[supported & (pd.to_numeric(df["suite_cycles"]) > parsed_budget)]
    assert over.empty, f"Hard cycle-budget violation in {root}:\n{over[['task_id', 'method', 'suite_cycles']]}"
    for name in ["budget_curve_summary.csv", "aukc_summary.csv", "paired_bootstrap.csv", "experiment_config.json", "experiment_fingerprint.txt"]:
        assert (root / name).exists(), f"Missing {name} in {root}"
    if require_llm:
        runtime = json.loads((root / "llm_runtime.json").read_text())
        devices = {str(v) for v in runtime.get("device_map", {}).values()}
        assert "cuda:0" in devices and "cuda:1" in devices, f"Model did not use both T4s: {devices}"
        assert not ({"cpu", "disk"} & devices), f"Forbidden model offload: {devices}"
    print(f"SANITY_VALIDATION_PASS: {root} | rows={len(df)}")
    return df

smoke_df = validate_sanity_run(
    "/kaggle/working/active_tb_plus_budget_smoke", expected_rows=54, allowed_budgets={8, 16}
)
if RUN_LLM_SMOKE:
    validate_sanity_run(
        "/kaggle/working/active_tb_plus_llm_smoke", expected_rows=2, allowed_budgets={16}, require_llm=True
    )

ablation_root = Path("/kaggle/working/active_tb_x_llm_ablation_v7")
ablation_df = validate_sanity_run(ablation_root, expected_rows=48, allowed_budgets={32}, require_llm=True)
state = json.loads((ablation_root / "run_state.json").read_text())
assert state.get("status") == "complete", state
assert Path("/kaggle/working/active_tb_x_llm_ablation_v7_checkpoint_latest.zip").exists()

summary = pd.read_csv(ablation_root / "budget_curve_summary.csv")
paired = pd.read_csv(ablation_root / "paired_bootstrap.csv")
direct = paired[
    (paired["method_a"] == "active_portfolio")
    & (paired["method_b"] == "active_portfolio_no_llm")
]
assert len(direct) == 1, f"Missing exact paired comparison:\n{paired}"
print("\n=== EXACT LLM ABLATION SUMMARY ===")
print(summary.to_string(index=False))
print("\n=== PAIRED TASK-LEVEL LLM CONTRIBUTION ===")
print(direct.to_string(index=False))

source_counts = Counter()
for path in ablation_root.rglob("b032__active_portfolio_suite.json"):
    for scenario in json.loads(path.read_text()):
        origin = scenario.get("metadata", {}).get("origin_source")
        source_counts[origin or scenario.get("source", "unknown")] += 1
print("\n=== FULL PORTFOLIO ACCEPTED SCENARIO ORIGINS ===")
print(dict(source_counts))

PRESERVE_DISPOSABLE_RUNTIME = False
if not PRESERVE_DISPOSABLE_RUNTIME:
    import os, shutil
    os.chdir("/kaggle/working")
    for disposable in [Path(VENV), Path(WORK_PROJECT), Path("/kaggle/working/active_tb_plus_attached_source")]:
        if disposable.exists():
            shutil.rmtree(disposable)
    print("DISPOSABLE_RUNTIME_CLEANUP_PASS")


## Resume policy

The runner checkpoints every completed `(task, method)` job and writes `active_tb_x_llm_ablation_v7_checkpoint_latest.zip`. If a session ends early, attach that ZIP to the next T4x2 session, set `BUDGET_SWEEP_CHECKPOINT`, and rerun. The experiment fingerprint prevents incompatible results from being mixed.
